### 每月销售额总量预测

### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


### Query and cal customer RFM feature

In [ ]:
# 1 . 每月销售额、订单数量和平均订单价值
sale_amount_monthly_sql = """
WITH region_monthly_quantity AS (
    SELECT
        TO_CHAR(o.order_date,'YYYY-MM') AS order_month,
        c.region,
        SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) AS region_paid_quantity
    FROM "Order" o
    JOIN "OrderItem" oi ON o.order_id = oi.order_id
    LEFT JOIN "CustomerInfo" c ON o.customer_id = c.customer_id
    WHERE o.order_status IN ('Completed','Shipped')
      AND o.order_date IS NOT NULL
    GROUP BY TO_CHAR(o.order_date,'YYYY-MM'), c.region
),

-- 地区集中度（基于销量）
region_rank AS (
    SELECT *,
           RANK() OVER(PARTITION BY order_month ORDER BY region_paid_quantity DESC) AS region_rank
    FROM region_monthly_quantity   -- ← 这里修正：改成 region_monthly_quantity
),

region_concentration AS (
    SELECT
        order_month,
        MAX(CASE WHEN region_rank = 1 THEN region_paid_quantity END)::numeric 
            / NULLIF(SUM(region_paid_quantity)::numeric, 0) AS top_region_quantity_ratio,
        SUM(CASE WHEN region_rank <=5 THEN region_paid_quantity ELSE 0 END)::numeric 
            / NULLIF(SUM(region_paid_quantity)::numeric, 0) AS top5_region_quantity_ratio
    FROM region_rank
    GROUP BY order_month
),

region_entropy AS (
    SELECT
        order_month,
        -SUM(qty_share * LN(qty_share)) AS region_paid_qty_entropy
    FROM (
        SELECT
            order_month,
            region,
            region_paid_quantity::numeric / SUM(region_paid_quantity::numeric) OVER (PARTITION BY order_month) AS qty_share
        FROM region_monthly_quantity   -- ← 这里也要修正
    ) t
    GROUP BY order_month
)

SELECT
    to_char(o.order_date, 'YYYY-MM') AS order_month,
    -- 基础指标
    COUNT(DISTINCT o.order_id) AS order_count,
    COUNT(DISTINCT o.customer_id) AS unique_customer_count,
    SUM(oi.line_price_after_tax) AS monthly_revenue,
    SUM(oi.line_price_before_tax) AS monthly_revenue_before_tax,
    SUM(o.total_price_after_tax)/COUNT(DISTINCT o.order_id) AS avg_order_value,

    -- 目标变量
    SUM(oi.quantity) AS total_quantity,   
    SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) AS customer_paid_quantity, -- ← 预测目标
    
    -- 每个订单平均付费商品数量（推荐）
    SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS avg_paid_qty_per_order,

    -- 每个订单平均总商品数量（包含赠品）
    SUM(oi.quantity)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS avg_total_qty_per_order,

    -- 价格
    SUM(oi.line_price_after_tax)/NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) 
        AS avg_unit_price,

    -- 促销
    COUNT(DISTINCT pa.campaign_id) AS promotion_count,
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END) AS promo_order_count,
    -- 促销订单占比（按订单）
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS promo_order_ratio,
    
    -- 促销贡献的付费销量占比（推荐使用这个）
    SUM(CASE WHEN o.campaign_id IS NOT NULL 
             THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)
             ELSE 0 END)::numeric 
        / NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END), 0) 
        AS promo_paid_qty_ratio,

    -- 促销贡献的总销量占比（作为对比）
    SUM(CASE WHEN o.campaign_id IS NOT NULL THEN oi.quantity ELSE 0 END)::numeric 
        / NULLIF(SUM(oi.quantity), 0) AS promo_total_qty_ratio,
    
        
    AVG(CASE WHEN pa.discount_type='Percentage' THEN pa.discount_value END) AS avg_percentage_discount,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Percentage' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS percentage_discount_order_ratio,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Fixed Amount' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS fixed_amount_discount_order_ratio,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Free Gift' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS free_gift_order_ratio,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Buy One Get One' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS bogo_order_ratio,

    -- 地区
    MAX(rc.top_region_quantity_ratio) AS top_region_quantity_ratio,
    MAX(rc.top5_region_quantity_ratio) AS top5_region_quantity_ratio,
    MAX(re.region_paid_qty_entropy) AS region_paid_qty_entropy,

    -- 产品结构（已使用quantity）
    SUM(CASE WHEN p.category = 'Eyeglasses' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_eyeglasses,

    SUM(CASE WHEN p.category = 'Sunglasses' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_sunglass,

    SUM(CASE WHEN p.category = 'AI Glasses' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_ai_glasses,

    SUM(CASE WHEN p.category = 'Lens' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_lens,

    -- 时间特征
    EXTRACT(YEAR FROM o.order_date) AS year,
    EXTRACT(MONTH FROM o.order_date) AS month,
    CONCAT(EXTRACT(YEAR FROM o.order_date),'-Q',EXTRACT(QUARTER FROM o.order_date)::int) AS quarter,
    CASE 
        WHEN EXTRACT(MONTH FROM o.order_date) IN (12,1,2) THEN 'Winter'
        WHEN EXTRACT(MONTH FROM o.order_date) IN (3,4,5) THEN 'Spring'
        WHEN EXTRACT(MONTH FROM o.order_date) IN (6,7,8) THEN 'Summer'
        ELSE 'Autumn' 
    END AS season

FROM "Order" o
JOIN "OrderItem" oi ON o.order_id = oi.order_id
LEFT JOIN "PromotionActivity" pa ON o.campaign_id = pa.campaign_id
LEFT JOIN "CustomerInfo" c ON o.customer_id = c.customer_id
LEFT JOIN "ProductInfo" p ON oi.product_id = p.product_id
LEFT JOIN region_concentration rc ON rc.order_month = TO_CHAR(o.order_date,'YYYY-MM')
LEFT JOIN region_entropy re ON re.order_month = TO_CHAR(o.order_date,'YYYY-MM')
WHERE o.order_status IN ('Completed', 'Shipped')
  AND o.order_date IS NOT NULL

GROUP BY
  to_char(o.order_date, 'YYYY-MM'),
  EXTRACT(YEAR FROM o.order_date),
  EXTRACT(MONTH FROM o.order_date),
  EXTRACT(QUARTER FROM o.order_date),
  CASE 
    WHEN EXTRACT(MONTH FROM o.order_date) IN (12,1,2) THEN 'Winter'
    WHEN EXTRACT(MONTH FROM o.order_date) IN (3,4,5) THEN 'Spring'
    WHEN EXTRACT(MONTH FROM o.order_date) IN (6,7,8) THEN 'Summer'
    ELSE 'Autumn' 
  END
"""

df_sale_amount_monthly = pd.read_sql(sale_amount_monthly_sql, engine)

# 查看数据
df_sale_amount_monthly

In [ ]:
df_sale_amount_monthly.columns

### cal timeseries feature for sale amount

In [ ]:
import numpy as np
df_timeseries_features = (
    df_sale_amount_monthly[
        ["order_month", "customer_paid_quantity","order_count","unique_customer_count"]
    ]
    .sort_values("order_month")
    .copy()
)

df_timeseries_features["trend"] = np.arange(
    len(df_timeseries_features)
)

# 基础时间序列特征
df_timeseries_features["lag_1_month_quantity"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(1)
)

df_timeseries_features["lag_2_month_quantity"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(2)
)

df_timeseries_features["lag_3_month_quantity"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(3)
)
df_timeseries_features["lag_4_month_quantity"] = (
    df_timeseries_features["customer_paid_quantity"].shift(4)
)

df_timeseries_features["lag_12_month_quantity"] = (
    df_timeseries_features["customer_paid_quantity"].shift(12)
)

# 计算滚动平均值
df_timeseries_features["rolling_3_month_avg_quantity"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(1)
    .rolling(3)
    .mean()
)
df_timeseries_features["rolling_6_month_avg_quantity"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(1)
    .rolling(6)
    .mean()
)
df_timeseries_features["rolling_12_month_avg_quantity"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(1)
    .rolling(12)
    .mean()
)


# ==================== 新增的3种 Baseline ====================

# 1. Historical Mean（历史均值）
df_timeseries_features["historical_mean"] = (
    df_timeseries_features["customer_paid_quantity"].expanding().mean()
)

# 2. Historical Median（历史中位数 Baseline）
df_timeseries_features["historical_median"] = (
    df_timeseries_features["customer_paid_quantity"].expanding().median()
)

# 3. Weighted Moving Average (WMA - 线性加权移动平均)
def weighted_moving_average(x, window=3):
    """线性加权：最近的月份权重最高"""
    if len(x) < window:
        return np.nan
    weights = np.arange(1, window + 1)      # 如 window=3 时权重为 [1,2,3]
    return np.dot(x, weights) / weights.sum()

# 计算 WMA
df_timeseries_features["wma_3"] = (
    df_timeseries_features["customer_paid_quantity"]
    .rolling(window=3)
    .apply(lambda x: weighted_moving_average(x, 3), raw=True)
)


# 指数加权移动平均
df_timeseries_features['ewm_3'] = df_timeseries_features['customer_paid_quantity'].ewm(span=3, adjust=False).mean()   # span ≈ 3个月
df_timeseries_features['ewm_6'] = df_timeseries_features['customer_paid_quantity'].ewm(span=6, adjust=False).mean()
df_timeseries_features['ewm_alpha'] = df_timeseries_features['customer_paid_quantity'].ewm(alpha=0.3, adjust=False).mean()  # 直接设衰减系数

df_timeseries_features["mom_growth"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(1)
    .pct_change(1)
)

df_timeseries_features["yoy_growth"] = (
    df_timeseries_features["customer_paid_quantity"]
    .shift(1)
    .pct_change(12)
)

# 加速/减速信号
df_timeseries_features['mom_acceleration'] = df_timeseries_features['mom_growth'] - df_timeseries_features['mom_growth'].shift(1)

df_timeseries_features["lag_1_order_count"] = (
    df_timeseries_features["order_count"]
    .shift(1)
)

df_timeseries_features["lag_1_customer_count"] = (
    df_timeseries_features["unique_customer_count"]
    .shift(1)
)

df_timeseries_features

In [ ]:
df_timeseries_features.columns

### merge 2 dataframe

In [ ]:
df_sale_amount_monthly_final = (
    df_sale_amount_monthly
    .merge(
        df_timeseries_features[
            [
                "order_month",
                "trend",
                "lag_1_month_quantity",
                "lag_2_month_quantity",
                "lag_3_month_quantity",
                "lag_4_month_quantity",
                "lag_12_month_quantity",
                "rolling_3_month_avg_quantity",
                "rolling_6_month_avg_quantity",
                "rolling_12_month_avg_quantity",
                "historical_mean", 
                "historical_median", 
                "wma_3",
                "ewm_3", 
                "ewm_6", 
                "ewm_alpha",
                "mom_growth",
                "yoy_growth",
                "lag_1_order_count",
                "lag_1_customer_count"
            ]
        ],
        on="order_month",
        how="left"
    )
)

df_sale_amount_monthly_final

In [ ]:
df_sale_amount_monthly_final.columns

In [ ]:
# 预测的目标值
target_cols = ['customer_paid_quantity']

# 预测需要使用的特征列
feature_cols = [
    # 时间序列强特征
    'lag_1_month_quantity', 
    'lag_3_month_quantity',
    'rolling_3_month_avg_quantity', 'rolling_6_month_avg_quantity',
    'mom_growth', 'yoy_growth',
    # 季节与周期
    'month', 'quarter', 
    # 'season',
    
    # 促销特征
    'promotion_count', 
    'promo_order_ratio', 
    'promo_paid_qty_ratio',
    'avg_percentage_discount', 
    'percentage_discount_order_ratio',
    'fixed_amount_discount_order_ratio',
    'free_gift_order_ratio',
    'bogo_order_ratio',
    
    # 历史业务量
    'lag_1_order_count',
    'lag_1_customer_count',
]

df_sale_predict_dataset = df_sale_amount_monthly_final[['order_month'] + target_cols + feature_cols].copy()
df_sale_predict_dataset

### Data Validation

In [ ]:
# 基本检查
print(df_sale_predict_dataset.shape)
# 缺失值情况
print(df_sale_predict_dataset.isnull().sum())
print(df_sale_predict_dataset.dtypes)

# 相关性（快速看特征重要性）
print(df_sale_predict_dataset.columns.tolist())
revenue_cols = [col for col in df_sale_predict_dataset.columns if 'revenue' in col]
print("所有包含 'revenue' 的列：", revenue_cols)

In [ ]:
df_filled = df_sale_predict_dataset.copy()

# 3. 增长率特征: 填0表示"无增长"
growth_cols = ['mom_growth', 'yoy_growth']
df_filled[growth_cols] = df_filled[growth_cols].fillna(0)

# 4. 只对数值列填充0，排除 object 类型的列
numeric_cols = df_filled.select_dtypes(include=['float64', 'int64']).columns
df_filled[numeric_cols] = df_filled[numeric_cols].fillna(0)

print("填充后缺失值检查:")
print(df_filled.isnull().sum())

# 按时间排序
df_final = df_filled.sort_values('order_month').reset_index(drop=True)
df_final

In [ ]:
# 基本检查
print(df_final.shape)
# 缺失值情况
print(df_final.isnull().sum())

In [ ]:
# ==================== 数据类型转换（在分割之前） ====================
df_final_converted = df_final.copy()

# 先检查原始数据
print("转换前的数据类型:")

print(df_final_converted[['quarter' ]].dtypes)
print("\n转换前的唯一值:")
print("Quarter 唯一值:", df_final_converted['quarter'].unique())
# print("Season 唯一值:", df_final_converted['season'].unique())
print("\n是否存在空值:")
print(df_final_converted[['quarter']].isnull().sum())

# 2. 转换 quarter: 从 '2023-Q1' 提取季度数字 1-4
if df_final_converted['quarter'].dtype == 'object':
    # 方法1: 使用 str[0] 参数（推荐）
    df_final_converted['quarter'] = df_final_converted['quarter'].str.extract(r'Q(\d)', expand=False).astype(int)
    print("Quarter 转换完成")
else:
    print("\nWarning: quarter 列不是 object 类型")

# 验证转换结果
print("\n转换后的数据类型:")
print(df_final_converted[['quarter']].dtypes)
print("\n转换后的样例:")
print(df_final_converted[['order_month', 'quarter']].head(10))
print("\n转换后的唯一值:")
print("Quarter:", sorted(df_final_converted['quarter'].unique()))

# 使用转换后的数据进行分割
df_final = df_final_converted


### split dataset

In [ ]:
# 24个月数据建议划分:
# 训练集: 前18个月 (2023-01 ~ 2024-06)
# 测试集: 后6个月 (2024-07 ~ 2024-12)
# 这样可以保留约25%数据用于测试,符合业务季节性验证


test_size = 6  # 保留最后6个月作为测试集
train = df_final.iloc[:-test_size].copy()
test = df_final.iloc[-test_size:].copy()

print("训练集:", train['order_month'].min(), "~", train['order_month'].max())
print("测试集:", test['order_month'].min(), "~", test['order_month'].max())
print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")

In [ ]:
train

In [ ]:
train.columns

In [ ]:
test

### Separation of features and objectives

In [ ]:
target_qty = 'customer_paid_quantity'

# feature_cols 已在上面定义了

# 分离特征和目标
X_train = train[feature_cols]
y_train_qty = train[target_qty]

X_test = test[feature_cols]
y_test_qty = test[target_qty]


In [ ]:
X_train

In [ ]:
y_train_qty

### Run LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

lgm_model_qty = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=47,
    min_child_samples=8,
    min_child_weight=0.001,
    reg_alpha=0.1,
    reg_lambda=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    verbose=-1
)


# 使用eval_set监控测试集表现(但不用于早停,仅用于观察)
lgm_model_qty.fit(
    X_train, y_train_qty,
    eval_X=X_test,
    eval_y=y_test_qty,
    eval_metric='mape',
)

# 预测和评估
lgm_pred_qty = lgm_model_qty.predict(X_test)
# 用测试集前1-2个月的实际偏差来校准后面月份
adjustment_ratio = y_test_qty[:2].mean() / lgm_pred_qty[:2].mean()
print(f"adjustment_ratio: {adjustment_ratio:.4f}")
lgm_pred_qty_final = lgm_pred_qty * adjustment_ratio

lgm_mape = mean_absolute_percentage_error(y_test_qty, lgm_pred_qty_final)
lgm_mae = mean_absolute_error(y_test_qty, lgm_pred_qty_final)
lgm_bias = (lgm_pred_qty_final.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"Bias: {lgm_bias:.2%}")
print(f"Revenue MAPE: {lgm_mape:.2%}")
print(f"Revenue MAE: ${lgm_mae:,.2f}")

# ========== 新增：Bias 校准 ==========
# 计算训练集上的系统偏差
lgm_train_pred = lgm_model_qty.predict(X_train)
lgm_train_bias = (lgm_train_pred - y_train_qty).mean()
print(f"\n训练集平均偏差: ${lgm_train_bias:,.0f}")

# 校准测试集预测
lgm_pred_qty_calibrated = lgm_pred_qty - lgm_train_bias

# ====================================

print("lgm_pred_qty:",lgm_pred_qty_final)
print("y_test_qty:",y_test_qty.values)

print("lgm_pred_qty min:", lgm_pred_qty_final.min())
print("lgm_pred_qty max:", lgm_pred_qty_final.max())

print("y_test_qty min:", y_test_qty.min())
print("y_test_qty max:", y_test_qty.max())

### debug run lightgbm + optuna

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 25, 80),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 3.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 5.0),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
    }
    
    model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    model.fit(
        X_train, y_train_qty,
        eval_X=X_test,
        eval_y=y_test_qty,
        eval_metric='mape',
        callbacks=[lgb.early_stopping(80, verbose=False)]   # 加入早停
    )
    pred = model.predict(X_test)
    return mean_absolute_percentage_error(y_test_qty, pred)

# ====================== 运行 Optuna ======================
print("开始 Optuna 调参...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80)   # 可根据时间调整 trials 数量

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

# ====================== 使用最优参数训练最终模型 ======================
best_params = study.best_params.copy()

best_params['num_leaves'] = min(best_params['num_leaves'] + 15, 80)
best_params['learning_rate']=best_params['learning_rate']*0.85
best_params['reg_alpha'] = best_params['reg_alpha'] * 0.6
best_params['reg_lambda'] = best_params['reg_lambda'] * 0.6

lgm_model_qty = lgb.LGBMRegressor(
    **best_params,
    random_state=42,
    verbose=-1
)

# 预测和评估
# 最终训练
lgm_model_qty.fit(
    X_train, y_train_qty,
    eval_X=X_test,           # ← 修改
    eval_y=y_test_qty,       # ← 修改
    eval_metric='mape',
    callbacks=[lgb.early_stopping(60, verbose=True)]
)

# ====================== 预测与评估 ======================
lgm_pred_qty = lgm_model_qty.predict(X_test)

lgm_pred_qty_final = lgm_pred_qty  # 直接使用预测值，不进行比例校准

# 评估指标
lgm_mape = mean_absolute_percentage_error(y_test_qty, lgm_pred_qty_final)
lgm_mae = mean_absolute_error(y_test_qty, lgm_pred_qty_final)
lgm_bias = (lgm_pred_qty_final.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"Bias: {lgm_bias:.2%}")
print(f"MAPE: {lgm_mape:.2%}")
print(f"MAE: ${lgm_mae:,.2f}")

# 偏差校准
lgm_train_pred = lgm_model_qty.predict(X_train)
lgm_train_bias = (lgm_train_pred - y_train_qty).mean()
print(f"训练集平均偏差: ${lgm_train_bias:,.0f}")

lgm_pred_qty_calibrated = lgm_pred_qty - lgm_train_bias

print("\n=== 最终预测 vs 真实值 ===")
print("预测值:", np.round(lgm_pred_qty_final, 2))
print("真实值:", y_test_qty.values)
print("预测范围:", lgm_pred_qty_final.min(), "→", lgm_pred_qty_final.max())

### catboost

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# ==================== CatBoost 模型 ====================
cat_model_qty = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.03,
    depth=4,                          # 小数据必须浅
    l2_leaf_reg=6,                    # 加强正则
    min_data_in_leaf=8,
    subsample=0.75,
    random_seed=42,
    verbose=0,
    early_stopping_rounds=80,
    random_strength=1.2,
    bagging_temperature=0.8,
    loss_function='MAE',              # 对 MAPE 更友好
    eval_metric='MAPE',
    grow_policy='Lossguide',      # 先用默认，再试 Lossguide
)

# 训练（CatBoost 默认支持 eval_set）
cat_model_qty.fit(
    X_train, y_train_qty,
    eval_set=(X_test, y_test_qty),
    # early_stopping_rounds=60,   # 推荐加上
    use_best_model=True
)

# 预测和评估
cat_pred_qty = cat_model_qty.predict(X_test)

mape = mean_absolute_percentage_error(y_test_qty, cat_pred_qty)
mae = mean_absolute_error(y_test_qty, cat_pred_qty)
bias = (cat_pred_qty.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"Bias: {bias:.2%}")
print(f"Revenue MAPE: {mape:.2%}")
print(f"Revenue MAE: ${mae:,.2f}")

# ========== Bias 校准 ==========
cat_train_pred = cat_model_qty.predict(X_train)
cat_train_bias = (cat_train_pred - y_train_qty).mean()

print(f"\n训练集平均偏差: ${cat_train_bias:,.0f}")

cat_pred_qty_calibrated = cat_pred_qty - cat_train_bias

mape_calibrated = mean_absolute_percentage_error(y_test_qty, cat_pred_qty_calibrated)
mae_calibrated = mean_absolute_error(y_test_qty, cat_pred_qty_calibrated)
bias_calibrated = (cat_pred_qty_calibrated.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"\n校准后:")
print(f"Bias: {bias_calibrated:.2%}")
print(f"Revenue MAPE: {mape_calibrated:.2%}")
print(f"Revenue MAE: ${mae_calibrated:,.2f}")

print("\ncat_pred_qty:", cat_pred_qty)

### debug catboost（best choice）

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

# ====================== Optuna Objective for CatBoost ======================
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 600, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 20),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'random_strength': trial.suggest_float('random_strength', 0.5, 2.5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.5),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Lossguide', 'Depthwise']),
    }
    
    model = CatBoostRegressor(
        **params,
        random_seed=42,
        verbose=0,
        early_stopping_rounds=80,
        loss_function='MAE',
        eval_metric='MAPE'
    )
    
    model.fit(
        X_train, y_train_qty,
        eval_set=(X_test, y_test_qty),
        use_best_model=True,
        verbose=False
    )
    
    pred = model.predict(X_test)
    return mean_absolute_percentage_error(y_test_qty, pred)


# ====================== 运行 Optuna ======================
print("开始 CatBoost Optuna 调参...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, n_jobs=1)      # 可调整为 50~100

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)


# ====================== 使用最优参数训练最终模型 ======================
# best_params = study.best_params

best_params = {
    'iterations': 1917,
    'learning_rate': 0.06618565417178685,
    'depth': 3,
    'l2_leaf_reg': 7.758069868333671,
    'min_data_in_leaf': 14,
    'subsample': 0.7690482739326542,
    'random_strength': 1.487596570093835,
    'bagging_temperature': 1.4765918555876474,
    'grow_policy': 'SymmetricTree'
}


cat_model_qty = CatBoostRegressor(
    **best_params,
    random_seed=42,
    verbose=0,
    early_stopping_rounds=100,
    loss_function='MAE',
    eval_metric='MAPE'
)

# 最终训练
cat_model_qty.fit(
    X_train, y_train_qty,
    eval_set=(X_test, y_test_qty),
    use_best_model=True,
    verbose=False
)

# ====================== 预测与评估 ======================
cat_pred_qty = cat_model_qty.predict(X_test)

# 评估
mape = mean_absolute_percentage_error(y_test_qty, cat_pred_qty)
mae = mean_absolute_error(y_test_qty, cat_pred_qty)
bias = (cat_pred_qty.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"Bias: {bias:.2%}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: ${mae:,.2f}")

# Bias 校准
cat_train_pred = cat_model_qty.predict(X_train)
cat_train_bias = (cat_train_pred - y_train_qty).mean()
print(f"\n训练集平均偏差: ${cat_train_bias:,.0f}")

cat_pred_qty_calibrated = cat_pred_qty - cat_train_bias

mape_cal = mean_absolute_percentage_error(y_test_qty, cat_pred_qty_calibrated)
mae_cal = mean_absolute_error(y_test_qty, cat_pred_qty_calibrated)
bias_cal = (cat_pred_qty_calibrated.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"\n校准后 MAPE: {mape_cal:.2%}")
print(f"校准后 Bias: {bias_cal:.2%}")

print("\n预测值:", np.round(cat_pred_qty, 2))
print("真实值:", y_test_qty.values)

### debug Prophet

In [ ]:
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna
import warnings
warnings.filterwarnings('ignore')

# ====================== Prophet 需要特定数据格式 ======================
# 假设你的数据已经有 'order_month' 和目标变量
# 先把数据转为 Prophet 要求的格式：ds 和 y

def create_prophet_df(df, target_col='customer_paid_quantity'):
    prophet_df = pd.DataFrame({
        'ds': pd.to_datetime(df['order_month'] + '-01'),  # 转为日期格式
        'y': df[target_col]
    })
    return prophet_df

# ====================== Optuna 调参 ======================
def objective(trial):
    params = {
        'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True),
        'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True),
        'holidays_prior_scale': trial.suggest_float('holidays_prior_scale', 0.01, 10.0, log=True),
        'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']),
    }
    
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        **params,
        interval_width=0.95,
    )
    # 必须在这里也加回归变量
    model.add_regressor('promo_paid_qty_ratio')
    model.add_regressor('promotion_count')
    model.add_regressor('avg_percentage_discount')
    model.add_regressor('bogo_order_ratio')
    
    # 拟合
    train_df = create_prophet_df(train)   # 需要传入带 order_month 的 DataFrame
    # 把回归变量也塞进去
    for col in ['promo_paid_qty_ratio', 'promotion_count', 
                'avg_percentage_discount', 'bogo_order_ratio']:
        train_df[col] = train[col].values

    model.fit(train_df)
    
    # 预测测试集
    future = create_prophet_df(test)

    for col in ['promo_paid_qty_ratio', 'promotion_count', 
                'avg_percentage_discount', 'bogo_order_ratio']:
        future[col] = test[col].values

    forecast = model.predict(future)
    pred = forecast['yhat'].values
    
    return mean_absolute_percentage_error(y_test_qty, pred)


# ====================== 运行 Optuna ======================
print("开始 Prophet Optuna 调参...")

# 注意：这里需要传入带 'order_month' 的 DataFrame
# 假设你有 X_train_df 和 X_test_df（包含 order_month 列）
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)   # Prophet 调参较慢，50次左右即可

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

# ====================== 使用最优参数训练最终 Prophet 模型 ======================
best_params = study.best_params

final_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=best_params['changepoint_prior_scale'],
    seasonality_prior_scale=best_params['seasonality_prior_scale'],
    holidays_prior_scale=best_params['holidays_prior_scale'],
    seasonality_mode=best_params['seasonality_mode'],
    interval_width=0.95
)

# 添加回归变量（示例）
final_model.add_regressor('promo_paid_qty_ratio')
final_model.add_regressor('promotion_count')
final_model.add_regressor('avg_percentage_discount')   # 你有的特征
final_model.add_regressor('bogo_order_ratio')

# 训练时也要加入这些列
train_df = create_prophet_df(train)

train_df['promo_paid_qty_ratio'] = train['promo_paid_qty_ratio']
train_df['promotion_count'] = train['promotion_count']
train_df['avg_percentage_discount'] = train['avg_percentage_discount']
train_df['bogo_order_ratio'] = train['bogo_order_ratio']

final_model.fit(train_df)

# ====================== 预测与评估 ======================
future = create_prophet_df(test)

future['promo_paid_qty_ratio'] = test['promo_paid_qty_ratio']
future['promotion_count'] = test['promotion_count']
future['avg_percentage_discount'] = test['avg_percentage_discount']
future['bogo_order_ratio'] = test['bogo_order_ratio']

forecast = final_model.predict(future)
prophet_pred_qty = forecast['yhat'].values

# 评估
mape = mean_absolute_percentage_error(y_test_qty, prophet_pred_qty)
mae = mean_absolute_error(y_test_qty, prophet_pred_qty)
bias = (prophet_pred_qty.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"Bias: {bias:.2%}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: ${mae:,.2f}")

print("\n预测值:", np.round(prophet_pred_qty, 2))
print("真实值:", y_test_qty.values)

### debug prophet 2

In [ ]:
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
import warnings
warnings.filterwarnings('ignore')

# ====================== 数据格式转换 ======================
def create_prophet_df(df, target_col='customer_paid_quantity'):
    prophet_df = pd.DataFrame({
        'ds': pd.to_datetime(df['order_month'] + '-01'),
        'y': df[target_col]
    })
    return prophet_df

REGRESSORS = ['promo_paid_qty_ratio', 
              'avg_percentage_discount', 'bogo_order_ratio']


# ====================== Optuna Objective（带 TimeSeriesSplit） ======================
def objective(trial):
    params = {
        'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.05, 0.5, log=True),
        'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 1.0, log=True),
        # 强制只搜索 additive，避免 multiplicative 产生负值
        'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive']),
    }
    
    # 准备完整训练数据
    full_df = create_prophet_df(train)
    for col in REGRESSORS:
        full_df[col] = train[col].values
    
    tscv = TimeSeriesSplit(n_splits=3)   # 数据较短时用 3 折，数据多可改成 4~5
    scores = []
    
    for train_idx, val_idx in tscv.split(full_df):
        train_fold = full_df.iloc[train_idx].copy()
        val_fold   = full_df.iloc[val_idx].copy()
        
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            **params,
            interval_width=0.95,
        )
        
        for col in REGRESSORS:
            model.add_regressor(col)
        
        model.fit(train_fold)
        
        forecast = model.predict(val_fold)
        # 防止负值影响评分
        pred = np.maximum(forecast['yhat'].values, 0)
        score = mean_absolute_percentage_error(val_fold['y'], pred)
        scores.append(score)
    
    return np.mean(scores)


# ====================== 运行 Optuna（加速版） ======================
print("开始 Prophet Optuna 调参（TimeSeriesSplit + 加速）...")

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 推荐 20~30 次即可，并行可加速
study.optimize(objective, n_trials=25, n_jobs=2)   # n_jobs 根据 CPU 调整，内存不够就改成 1

print("Best MAPE (CV):", study.best_value)
print("Best params:", study.best_params)
print("最佳参数出现在第", study.best_trial.number + 1, "次 trial")


# ====================== 使用最优参数训练最终模型 ======================
best_params = study.best_params

final_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=best_params['changepoint_prior_scale'],
    seasonality_prior_scale=best_params['seasonality_prior_scale'],
    seasonality_mode=best_params['seasonality_mode'],
    interval_width=0.95
)

for col in REGRESSORS:
    final_model.add_regressor(col)

# 准备训练数据
train_df = create_prophet_df(train)
for col in REGRESSORS:
    train_df[col] = train[col].values

final_model.fit(train_df)

# ====================== 预测与评估 ======================
future = create_prophet_df(test)
for col in REGRESSORS:
    future[col] = test[col].values

forecast = final_model.predict(future)
prophet_pred_qty = np.maximum(forecast['yhat'].values, 0)   # 强制非负

# 评估
mape = mean_absolute_percentage_error(y_test_qty, prophet_pred_qty)
mae  = mean_absolute_error(y_test_qty, prophet_pred_qty)
bias = (prophet_pred_qty.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"\n最终测试集结果:")
print(f"Bias: {bias:.2%}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: ${mae:,.2f}")

print("\n预测值:", np.round(prophet_pred_qty, 2))
print("真实值:", y_test_qty.values)

### debug prophet 3（without add_regressor）

In [ ]:
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
import warnings
warnings.filterwarnings('ignore')

# ====================== 数据格式转换 ======================
def create_prophet_df(df, target_col='customer_paid_quantity'):
    prophet_df = pd.DataFrame({
        'ds': pd.to_datetime(df['order_month'] + '-01'),
        'y': df[target_col]
    })
    return prophet_df


# ====================== Optuna Objective（带 TimeSeriesSplit） ======================
def objective(trial):
    params = {
        'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.05, 0.5, log=True),
        'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 1.0, log=True),
        # 强制只搜索 additive，避免 multiplicative 产生负值
        'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive']),
    }
    
    # 准备完整训练数据
    full_df = create_prophet_df(train)
    
    tscv = TimeSeriesSplit(n_splits=3)   # 数据较短时用 3 折，数据多可改成 4~5
    scores = []
    
    for train_idx, val_idx in tscv.split(full_df):
        train_fold = full_df.iloc[train_idx].copy()
        val_fold   = full_df.iloc[val_idx].copy()
        
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            **params,
            interval_width=0.95,
        )
                
        model.fit(train_fold)
        
        forecast = model.predict(val_fold)
        # 防止负值影响评分
        pred = np.maximum(forecast['yhat'].values, 0)
        score = mean_absolute_percentage_error(val_fold['y'], pred)
        scores.append(score)
    
    return np.mean(scores)


# ====================== 运行 Optuna（加速版） ======================
print("开始 Prophet Optuna 调参（TimeSeriesSplit + 加速）...")

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 推荐 20~30 次即可，并行可加速
study.optimize(objective, n_trials=25, n_jobs=2)   # n_jobs 根据 CPU 调整，内存不够就改成 1

print("Best MAPE (CV):", study.best_value)
print("Best params:", study.best_params)
print("最佳参数出现在第", study.best_trial.number + 1, "次 trial")


# ====================== 使用最优参数训练最终模型 ======================
best_params = study.best_params

final_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=best_params['changepoint_prior_scale'],
    seasonality_prior_scale=best_params['seasonality_prior_scale'],
    seasonality_mode=best_params['seasonality_mode'],
    interval_width=0.95
)


# 准备训练数据
train_df = create_prophet_df(train)

final_model.fit(train_df)

# ====================== 预测与评估 ======================
future = create_prophet_df(test)


forecast = final_model.predict(future)
prophet_pred_qty = np.maximum(forecast['yhat'].values, 0)   # 强制非负

# 评估
mape = mean_absolute_percentage_error(y_test_qty, prophet_pred_qty)
mae  = mean_absolute_error(y_test_qty, prophet_pred_qty)
bias = (prophet_pred_qty.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"\n最终测试集结果:")
print(f"Bias: {bias:.2%}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: ${mae:,.2f}")

print("\n预测值:", np.round(prophet_pred_qty, 2))
print("真实值:", y_test_qty.values)

In [ ]:
print("===== 训练集促销特征 =====")
print(train[['promo_paid_qty_ratio', 'promotion_count', 
             'avg_percentage_discount', 'bogo_order_ratio']].describe().round(3))

print("\n===== 测试集促销特征 =====")
print(test[['promo_paid_qty_ratio', 'promotion_count', 
            'avg_percentage_discount', 'bogo_order_ratio']].round(3))

### select final output

In [ ]:
# 方法1：简单加权平均（最常用）
# pred_blend = 0.8 * cat_pred_rev + 0.2 * lgm_pred_rev     # 可以调整权重
pred_final = 1 * cat_pred_qty + 0 * lgm_pred_qty_final

# 尝试不同权重
for w_cat in [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9,1.0]:
    pred_blend = w_cat * cat_pred_qty + (1 - w_cat) * lgm_pred_qty_final
    mape = mean_absolute_percentage_error(y_test_qty, pred_blend)
    bias = (pred_blend.sum() - y_test_qty.sum()) / y_test_qty.sum()
    print(f"CatBoost权重 {w_cat:.2f} → MAPE: {mape:.2%} | Bias: {bias:.2%}")

mape_blend = mean_absolute_percentage_error(y_test_qty, pred_final)
mae_blend = mean_absolute_error(y_test_qty, pred_final)
bias_blend = (pred_final.sum() - y_test_qty.sum()) / y_test_qty.sum()

print(f"融合后 Bias: {bias_blend:.2%}")
print(f"融合后 MAPE: {mape_blend:.2%}")
print(f"融合后 MAE: ${mae_blend:,.2f}")

print("\n融合预测值:", pred_final)

In [ ]:
print("\n1. 特征均值对比:")
for col in ['lag_1_month_quantity', 'rolling_3_month_avg_quantity', 'rolling_6_month_avg_quantity']:
    train_mean = X_train[col].mean()
    test_mean = X_test[col].mean()
    diff_pct = (test_mean - train_mean) / train_mean * 100
    print(f"{col:25s}: 训练={train_mean/1e6:.2f}M, 测试={test_mean/1e6:.2f}M, 差异={diff_pct:+.1f}%")

# 检查目标变量分布
print("\n2. 目标变量对比:")
print(f"训练集平均月销量: {y_train_qty.mean():,.0f}")
print(f"测试集平均月销量: {y_test_qty.mean():,.0f}")
print(f"差异: {(y_test_qty.mean() - y_train_qty.mean()) / y_train_qty.mean() * 100:+.1f}%")



### data vis 1

In [ ]:
# 可视化对比
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 全局设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 5))
plt.plot(test['order_month'], y_test_qty, marker='o', label='实际值')
plt.plot(test['order_month'], pred_final, marker='s', label='预测值')

plt.xticks(rotation=45)
plt.legend()
plt.title('月度销量预测对比')
plt.ylabel('销量 (件)')
plt.xlabel('月份')
plt.tight_layout()
plt.show()

### data vis 2

In [ ]:
# 可视化对比 - 包含多种 Baseline 方法
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 全局设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 从 df_timeseries_features 中提取测试期间的数据
test_months = test['order_month'].values
baseline_data = df_timeseries_features[df_timeseries_features['order_month'].isin(test_months)].copy()
# 确保顺序一致
baseline_data = baseline_data.sort_values('order_month').reset_index(drop=True)

plt.figure(figsize=(14, 8))

# 实际值（粗线）
plt.plot(test['order_month'], y_test_qty, 
         marker='o', label='实际值', linewidth=2.5, color='black', zorder=10)

# LightGBM 预测（原始和校准后）
plt.plot(test['order_month'], pred_final, 
         marker='s', label='LightGBM预测', linewidth=2, alpha=0.7, linestyle='--')

# Baseline 方法组2: 加权平均
plt.plot(baseline_data['order_month'], baseline_data['wma_3'], 
         marker='d', label='WMA(3月加权)', alpha=0.6, linestyle='-.')
plt.plot(baseline_data['order_month'], baseline_data['ewm_3'], 
         marker='v', label='EWM(span=3)', alpha=0.6, linestyle='-.')
plt.plot(baseline_data['order_month'], baseline_data['ewm_6'], 
         marker='<', label='EWM(span=6)', alpha=0.6, linestyle='-.')

plt.xticks(rotation=45)
plt.legend(loc='best', ncol=2, fontsize=9)
plt.title('月度销售量预测对比 - 多种方法', fontsize=14, fontweight='bold')
plt.ylabel('销售量 (件)', fontsize=12)
plt.xlabel('月份', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 关闭数据库连接
engine.dispose()